In [3]:
import os
from pyspark.sql import SparkSession

# Khởi tạo Spark Session cho Lab 4
spark = (SparkSession.builder
    .appName("Lab4_SparkML_Home")
    # Nạp gói Kafka (bắt buộc để làm Ex 0)
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1")
    # Kích hoạt AQE theo đặc tả trang 2 của Lab 4
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .master("local[*]") 
    .getOrCreate())

print("--- Spark Session cho Lab 4 đã sẵn sàng! ---")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/14 10:43:28 WARN Utils: Your hostname, HungThinh, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/14 10:43:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/thinh/BigData/bigdata_lab/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/thinh/.ivy2.5.2/cache
The jars for the packages stored in: /home/thinh/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b7edbff6-6c21-4a7d-80b0-fccf9f46eec8;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	foun

--- Spark Session cho Lab 4 đã sẵn sàng! ---


In [4]:
from confluent_kafka.admin import AdminClient, NewTopic

admin_client = AdminClient({'bootstrap.servers': 'localhost:9092,localhost:9192,localhost:9292'})
topic_names = ['Lab1_movies', 'Lab1_ratings', 'Lab1_tags']
admin_client.delete_topics(topic_names, operation_timeout=10)
new_topics = [NewTopic(topic=name, num_partitions=1, replication_factor=1) for name in topic_names]
fs = admin_client.create_topics(new_topics)

for topic, f in fs.items():
    try:
        f.result()
        print(f"Topic '{topic}' đã được khởi tạo thành công!")
    except Exception as e:
        print(f"Bỏ qua/Lỗi tại {topic}: {e}")

Topic 'Lab1_movies' đã được khởi tạo thành công!
Topic 'Lab1_ratings' đã được khởi tạo thành công!
Topic 'Lab1_tags' đã được khởi tạo thành công!


In [5]:
import kagglehub
from pyspark.sql.functions import to_json, struct

# 1. Tải dataset từ Kaggle
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

# 2. Đọc file CSV bằng Spark 
df_r = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_m = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_t = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)

# 3. Hàm đẩy dữ liệu vào Kafka
def push_to_kafka(df, topic):
    df.selectExpr("to_json(struct(*)) AS value") \
      .write.format("kafka") \
      .option("kafka.bootstrap.servers", "localhost:9092,localhost:9192,localhost:9292") \
      .option("topic", topic) \
      .save()
    print(f"Đã nạp dữ liệu cho {topic}")

# Thực hiện nạp cho 3 topic 
push_to_kafka(df_m, "Lab1_movies")
push_to_kafka(df_r, "Lab1_ratings")
push_to_kafka(df_t, "Lab1_tags")

/home/thinh/BigData/bigdata_lab/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đã nạp dữ liệu cho Lab1_movies


Đã nạp dữ liệu cho Lab1_ratings
Đã nạp dữ liệu cho Lab1_tags


In [14]:
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col

# 1. Định nghĩa Schema cho movies, ratings và tags theo đặc tả Lab 4
movie_schema = StructType([
    StructField("movieId", IntegerType()), 
    StructField("title", StringType()), 
    StructField("genres", StringType())
])
rating_schema = StructType([
    StructField("userId", IntegerType()), 
    StructField("movieId", IntegerType()), 
    StructField("rating", DoubleType()), 
    StructField("timestamp", LongType())
])
tag_schema = StructType([
    StructField("userId", IntegerType()), 
    StructField("movieId", IntegerType()), 
    StructField("tag", StringType()), 
    StructField("timestamp", LongType())
])

# 2. Hàm load dữ liệu từ Kafka vào DataFrame
def load_kafka(topic, schema):
    return (spark.read.format("kafka")
        .option("kafka.bootstrap.servers", "localhost:9092,localhost:9192,localhost:9292")
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .load()
        .select(from_json(col("value").cast("string"), schema).alias("data"))
        .select("data.*"))

# 3. Chuyển đổi dữ liệu sang định dạng chuẩn
df_movies = load_kafka("Lab1_movies", movie_schema)
df_ratings = load_kafka("Lab1_ratings", rating_schema)
df_tags = load_kafka("Lab1_tags", tag_schema)

print("--- Exercise 0: Hoàn thành! ---")
df_movies.show(5)
df_ratings.show(5)
df_tags.show(5)

--- Exercise 0: Hoàn thành! ---
+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows
+------+-------+---------------+----------+
|userId|movieId|            tag| timestamp|
+------+-------+---------------+----------+
|     2|  6075

Exercise1:

In [30]:
from pyspark.sql.functions import avg, count, when, col, concat_ws, collect_list, split, regexp_replace, lower

# 1. Gán nhãn: 1 nếu rating >= 4.0, ngược lại là 0
df_labeled = df_ratings.withColumn("label", when(col("rating") >= 4.0, 1).otherwise(0))

# 2. Numerics: Tính toán đầy đủ 4 chỉ số aggregates cho Movie và User [cite: 154, 155, 156]
movie_stats = df_ratings.groupBy("movieId").agg(
    avg("rating").alias("movie_avg_rating"),
    count("rating").alias("movie_rating_count")
)
user_stats = df_ratings.groupBy("userId").agg(
    avg("rating").alias("user_avg_rating"),
    count("rating").alias("user_rating_count")
)

# 3. Text: Gom nhóm Tags và chuẩn bị cột văn bản tổng hợp
user_movie_tags = df_tags.groupBy("userId", "movieId").agg(
    concat_ws(" ", collect_list("tag")).alias("user_tags")
)

# 4. Join tất cả và xử lý đa nhãn (Multi-label) cho Genres [cite: 352]
train_df = df_labeled.join(df_movies, "movieId") \
    .join(movie_stats, "movieId") \
    .join(user_stats, "userId") \
    .join(user_movie_tags, ["userId", "movieId"], "left") \
    .fillna("")

# Tách genres thành mảng các token để xử lý multi-label
train_df = train_df.withColumn("genres_tokens", split(col("genres"), "\\|"))

# LÀM SẠCH VĂN BẢN: Chuyển về chữ thường và xóa các ký tự đặc biệt/dấu câu
raw_text_col = concat_ws(" ", col("title"), col("user_tags"))
train_df = train_df.withColumn("text_clean", regexp_replace(lower(raw_text_col), r"[^\w\s]", ""))

print("Dữ liệu đã được làm sạch và chuẩn bị xong cho Exercise 1!")

Dữ liệu đã được làm sạch và chuẩn bị xong cho Exercise 1!


In [31]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, CountVectorizer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression

# Bước 1: Tokenizer và Vectorizer cho Văn bản (Title + Tags)
tokenizer = Tokenizer(inputCol="text_clean", outputCol="words")
cv_text = CountVectorizer(inputCol="words", outputCol="text_features")

# Bước 2: Vectorizer cho Thể loại (Multi-label Genres)
cv_genres = CountVectorizer(inputCol="genres_tokens", outputCol="genre_features")

# Bước 3: Gom nhóm toàn bộ đặc trưng vào 1 Vector duy nhất [cite: 306]
# Thứ tự quan trọng để mapping tên đặc trưng ở Cell 8
assembler = VectorAssembler(
    inputCols=[
        "text_features", 
        "genre_features", 
        "movie_avg_rating", "movie_rating_count", 
        "user_avg_rating", "user_rating_count"
    ],
    outputCol="raw_features"
)

# Bước 4: Chuẩn hóa dữ liệu và Khai báo mô hình [cite: 307, 309]
scaler = StandardScaler(inputCol="raw_features", outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)

# Tạo Pipeline hoàn chỉnh [cite: 311]
pipeline = Pipeline(stages=[tokenizer, cv_text, cv_genres, assembler, scaler, lr])

# Chia dữ liệu 80/20 và huấn luyện
train_data, test_data = train_df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train_data)

print("--- Mô hình Pipeline đã huấn luyện thành công! ---")

26/05/14 08:50:50 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/14 08:51:01 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/05/14 08:51:13 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...


--- Mô hình Pipeline đã huấn luyện thành công! ---


In [37]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# 1. Ẩn các cảnh báo Warning để đầu ra sạch sẽ
spark.sparkContext.setLogLevel("ERROR")

# 2. Dự đoán trên tập Test
predictions = model.transform(test_data)

# 3. Tính các chỉ số đo lường [cite: 541]
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(predictions)
f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(predictions)

print("="*50)
print(f"KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH")
print("-" * 50)
print(f"AUC: {auc:.4f}")
print(f"F1 Score:           {f1:.4f}")
print("="*50)

# 4. Tính toán các giá trị Ma trận nhầm lẫn [cite: 541]
tp = predictions.filter("prediction = 1.0 AND label = 1").count()
tn = predictions.filter("prediction = 0.0 AND label = 0").count()
fp = predictions.filter("prediction = 1.0 AND label = 0").count()
fn = predictions.filter("prediction = 0.0 AND label = 1").count()

# 5. In duy nhất một bảng Ma trận nhầm lẫn chuẩn báo cáo
print("\nCONFUSION MATRIX")
print("-" * 50)
print(f"{'':<15} | {'Dự đoán: 0':<12} | {'Dự đoán: 1':<12}")
print("-" * 50)
print(f"{'Thực tế: 0':<15} | {tn:<12} | {fp:<12} (TN | FP)")
print(f"{'Thực tế: 1':<15} | {fn:<12} | {tp:<12} (FN | TP)")
print("-" * 50)
print("Chú thích: TN: True Negative, TP: True Positive")
print("           FP: False Positive, FN: False Negative")

KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH
--------------------------------------------------
AUC: 0.7795
F1 Score:           0.7133



CONFUSION MATRIX
--------------------------------------------------
                | Dự đoán: 0   | Dự đoán: 1  
--------------------------------------------------
Thực tế: 0      | 7515         | 3004         (TN | FP)
Thực tế: 1      | 2806         | 6933         (FN | TP)
--------------------------------------------------
Chú thích: TN: True Negative, TP: True Positive
           FP: False Positive, FN: False Negative


In [43]:
import numpy as np

# 1. Truy xuất trọng số từ stage cuối cùng (Logistic Regression)
lr_model = model.stages[-1]
weights = lr_model.coefficients.toArray()

# 2. Lấy bộ từ vựng (Vocabulary) từ các stage Vectorizer
vocab_text = model.stages[1].vocabulary
vocab_genres = model.stages[2].vocabulary

# 3. Tạo danh sách tên đặc trưng theo đúng thứ tự đã đưa vào Assembler
all_feature_names = vocab_text + vocab_genres + [
    "movie_avg_rating", "movie_rating_count", 
    "user_avg_rating", "user_rating_count"
]

# 4. Sắp xếp để tìm các tín hiệu mạnh nhất
indices = np.argsort(weights)
print("TOP 10 POSITIVE SIGNALS")
print(f"{'FEATURE NAME':<25} | {'WEIGHT':<10}")
print("-" * 40)
for i in indices[-10:][::-1]:
    if i < len(all_feature_names):
        print(f"{all_feature_names[i]:<25} | {weights[i]:.4f}")

print("\n")
print("\nTOP 10 NEGATIVE SIGNALS")
print(f"{'FEATURE NAME':<25} | {'WEIGHT':<10}")
print("-" * 40)
for i in indices[:10]:
    if i < len(all_feature_names):
        print(f"{all_feature_names[i]:<25} | {weights[i]:.4f}")


TOP 10 POSITIVE SIGNALS
FEATURE NAME              | WEIGHT    
----------------------------------------
user_avg_rating           | 0.9618
movie_avg_rating          | 0.8097
movie_rating_count        | 0.1004
Documentary               | 0.0633
user_rating_count         | 0.0585
yojimbo                   | 0.0471
sexuality                 | 0.0437
cinematography            | 0.0413
1974                      | 0.0407
silly                     | 0.0384



TOP 10 NEGATIVE SIGNALS
FEATURE NAME              | WEIGHT    
----------------------------------------
godzilla                  | -0.0455
clockers                  | -0.0443
2008                      | -0.0426
batman                    | -0.0421
salvation                 | -0.0407
british                   | -0.0405
knowing                   | -0.0390
eli                       | -0.0381
kindergarten              | -0.0379
airheads                  | -0.0377


Exercise2:

In [46]:
from pyspark.sql.functions import collect_list, concat_ws, split, regexp_replace, lower, col

# 1. Gom tất cả tags của mỗi phim thành một chuỗi văn bản duy nhất
# Chúng ta dùng df_tags đã load từ Exercise 0
movie_tags_text = df_tags.groupBy("movieId").agg(
    concat_ws(" ", collect_list("tag")).alias("all_tags")
)

# 2. Join với df_movies để có thông tin Title và Genres
df_clustering = df_movies.join(movie_tags_text, "movieId", "left").fillna("")

# 3. Tiền xử lý văn bản tags (chuyển chữ thường, xóa dấu câu để TF-IDF chuẩn hơn)
df_clustering = df_clustering.withColumn("tags_clean", regexp_replace(lower(col("all_tags")), r"[^\w\s]", ""))

# 4. Xử lý Genres: Chuyển chuỗi "Action|Sci-Fi" thành mảng ["Action", "Sci-Fi"]
df_clustering = df_clustering.withColumn("genres_array", split(col("genres"), "\\|"))

print("--- Bước 1: Dữ liệu phân cụm đã sẵn sàng ---")
df_clustering.select("movieId", "title", "genres_array").show(5, truncate=False)

--- Bước 1: Dữ liệu phân cụm đã sẵn sàng ---
+-------+----------------------------------+-------------------------------------------------+
|movieId|title                             |genres_array                                     |
+-------+----------------------------------+-------------------------------------------------+
|1      |Toy Story (1995)                  |[Adventure, Animation, Children, Comedy, Fantasy]|
|2      |Jumanji (1995)                    |[Adventure, Children, Fantasy]                   |
|3      |Grumpier Old Men (1995)           |[Comedy, Romance]                                |
|4      |Waiting to Exhale (1995)          |[Comedy, Drama, Romance]                         |
|5      |Father of the Bride Part II (1995)|[Comedy]                                         |
+-------+----------------------------------+-------------------------------------------------+
only showing top 5 rows


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, CountVectorizer, IDF, VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# 1. Các thành phần Pipeline
tokenizer_cl = Tokenizer(inputCol="tags_clean", outputCol="words")

#Dùng CountVectorizer thay cho HashingTF để lấy tên từ khóa 
cv_tags_cl = CountVectorizer(inputCol="words", outputCol="raw_text_features")
idf_cl = IDF(inputCol="raw_text_features", outputCol="text_tfidf")

cv_genres_cl = CountVectorizer(inputCol="genres_array", outputCol="genre_features")

# 2. Gom đặc trưng Tags và Genres
assembler_cl = VectorAssembler(
    inputCols=["text_tfidf", "genre_features"], 
    outputCol="features"
)

# 3. Evaluator
evaluator_cl = ClusteringEvaluator(featuresCol="features", predictionCol="cluster", metricName="silhouette")

# 4. Vòng lặp thử nghiệm K = 6, 8, 10, 12
k_candidates = [6, 8, 10, 12] 
results = {}

for k in k_candidates:
    kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=k, seed=42)
    pipeline_cl = Pipeline(stages=[tokenizer_cl, cv_tags_cl, idf_cl, cv_genres_cl, assembler_cl, kmeans])
    
    cl_model = pipeline_cl.fit(df_clustering)
    cl_predictions = cl_model.transform(df_clustering)
    
    silhouette = evaluator_cl.evaluate(cl_predictions)
    results[k] = (cl_model, cl_predictions, silhouette)
    print(f"K = {k:2d} | Silhouette Score: {silhouette:.4f}")

K =  6 | Silhouette Score: 0.8019


K =  8 | Silhouette Score: 0.1176


K = 10 | Silhouette Score: 0.1929


K = 12 | Silhouette Score: 0.3460


In [62]:
import numpy as np
from pyspark.sql import functions as F

# Đảm bảo k_candidates đã được khai báo ở cell trước là [6, 8, 10, 12]
for k in k_candidates:
    print("\n" + "="*85)
    print(f" TỔNG QUAN VÀ CHI TIẾT PHÂN CỤM VỚI K = {k} ".center(85, "="))
    print("="*85)
    
    # 1. Lấy Model và kết quả dự đoán tương ứng với K
    cl_model = results[k][0]
    predictions_k = results[k][1]
    
    # --- PHẦN BỔ SUNG: BẢNG THỐNG KÊ TỔNG QUÁT ---
    print(f"\n Thống kê số lượng phim phân bổ trong {k} cụm:")
    # Hiển thị bảng count để thấy độ lệch của các cụm
    predictions_k.groupBy("cluster").count().orderBy("cluster").show()
    
    # 2. Lấy bộ từ vựng (Vocabulary) để ánh xạ tên từ khóa 
    # Stage 1: CountVectorizer của Tags | Stage 3: CountVectorizer của Genres
    vocab_tags = cl_model.stages[1].vocabulary
    vocab_genres = cl_model.stages[3].vocabulary
    all_vocab = vocab_tags + vocab_genres
    
    # 3. Truy xuất tâm cụm (Centroids) từ mô hình KMeans
    kmeans_model = cl_model.stages[-1]
    centers = kmeans_model.clusterCenters()
    
    # 4. Duyệt chi tiết từng cụm để in Từ khóa và Phim mẫu theo yêu cầu
    for cluster_id in range(k):
        print(f"\n>>> CHI TIẾT CỤM SỐ {cluster_id}")
        print("-" * 40)
        
        # A. TRÍCH XUẤT 10 TỪ KHÓA TIÊU BIỂU (Top terms)
        center = centers[cluster_id]
        # Lấy index của 10 trọng số lớn nhất trong tâm cụm
        top_indices = np.argsort(center)[-10:][::-1]
        
        # Ánh xạ index về tên từ/thể loại thực tế
        top_terms = [all_vocab[idx] for idx in top_indices if idx < len(all_vocab)]
        print(f"Từ khóa mô tả: {', '.join(top_terms)}")
        
        # B. XUẤT 10 PHIM MẪU TIÊU BIỂU (Sample movies)
        print("10 phim mẫu tiêu biểu:")
        predictions_k.filter(F.col("cluster") == cluster_id) \
            .select("title", "genres") \
            .limit(10) \
            .show(truncate=False)
            
    print("\n" + "."*85)


====================== TỔNG QUAN VÀ CHI TIẾT PHÂN CỤM VỚI K = 6 =====================

 Thống kê số lượng phim phân bổ trong 6 cụm:


+-------+-----+
|cluster|count|
+-------+-----+
|      0| 9724|
|      1|    1|
|      2|    1|
|      3|   13|
|      4|    2|
|      5|    1|
+-------+-----+


>>> CHI TIẾT CỤM SỐ 0
----------------------------------------
Từ khóa mô tả: Drama, Comedy, Thriller, Action, Romance, , Adventure, Crime, Horror, Sci-Fi
10 phim mẫu tiêu biểu:


+--------------------------------+--------------------------------+
|title                           |genres                          |
+--------------------------------+--------------------------------+
|Awfully Big Adventure, An (1995)|Drama                           |
|Hudsucker Proxy, The (1994)     |Comedy                          |
|What Happened Was... (1994)     |Comedy|Drama|Romance|Thriller   |
|High School High (1996)         |Comedy                          |
|Dirty Dancing (1987)            |Drama|Musical|Romance           |
|Local Hero (1983)               |Comedy                          |
|Candyman (1992)                 |Horror|Thriller                 |
|Men in Black (a.k.a. MIB) (1997)|Action|Comedy|Sci-Fi            |
|Spawn (1997)                    |Action|Adventure|Sci-Fi|Thriller|
|The Devil's Advocate (1997)     |Drama|Mystery|Thriller          |
+--------------------------------+--------------------------------+


>>> CHI TIẾT CỤM SỐ 1
------------------------

+-------------------+---------------------------+
|title              |genres                     |
+-------------------+---------------------------+
|Pulp Fiction (1994)|Comedy|Crime|Drama|Thriller|
+-------------------+---------------------------+


>>> CHI TIẾT CỤM SỐ 2
----------------------------------------
Từ khóa mô tả: reno, jean, assassin, hit, corruption, police, men, crime, lolita, besson
10 phim mẫu tiêu biểu:


+--------------------------------------------------------------+---------------------------+
|title                                                         |genres                     |
+--------------------------------------------------------------+---------------------------+
|Léon: The Professional (a.k.a. The Professional) (Léon) (1994)|Action|Crime|Drama|Thriller|
+--------------------------------------------------------------+---------------------------+


>>> CHI TIẾT CỤM SỐ 3
----------------------------------------
Từ khóa mô tả: visually, thoughtprovoking, appealing, surreal, comedy, dreamlike, atmospheric, philosophy, hallucinatory, cult
10 phim mẫu tiêu biểu:


...

+--------------------------------+----------------------------------------------+
|title                           |genres                                        |
+--------------------------------+----------------------------------------------+
|Men in Black (a.k.a. MIB) (1997)|Action|Comedy|Sci-Fi                          |
|Spawn (1997)                    |Action|Adventure|Sci-Fi|Thriller              |
|King Kong (1933)                |Action|Adventure|Fantasy|Horror               |
|Dungeons & Dragons (2000)       |Action|Adventure|Comedy|Fantasy               |
|Extreme Days (2001)             |Action|Adventure|Comedy|Drama                 |
|Big Doll House, The (1971)      |Action                                        |
|3:10 to Yuma (1957)             |Action|Adventure|Drama|Thriller|Western       |
|I Spy (2002)                    |Action|Adventure|Comedy|Crime                 |
|Atragon (Kaitei Gunkan) (1963)  |Adventure|Sci-Fi                              |
|Chase, The (199

+---------------------------------------------------------------------------+---------------------------+
|title                                                                      |genres                     |
+---------------------------------------------------------------------------+---------------------------+
|Wild Tales (2014)                                                          |Comedy|Drama|Thriller      |
|Big Lebowski, The (1998)                                                   |Comedy|Crime               |
|Man Bites Dog (C'est arrivé près de chez vous) (1992)                      |Comedy|Crime|Drama|Thriller|
|Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)|Comedy|War                 |
|Mary and Max (2009)                                                        |Animation|Comedy|Drama     |
|Burn After Reading (2008)                                                  |Comedy|Crime|Drama         |
+---------------------------------------------

+----------------+----------------------------+
|title           |genres                      |
+----------------+----------------------------+
|Star Trek (2009)|Action|Adventure|Sci-Fi|IMAX|
+----------------+----------------------------+


>>> CHI TIẾT CỤM SỐ 10
----------------------------------------
Từ khóa mô tả: soundtrack, humour, christoph, performances, waltz, samuel, jackson, l, western, quentin
10 phim mẫu tiêu biểu:


+-----------------------+--------------------+
|title                  |genres              |
+-----------------------+--------------------+
|Django Unchained (2012)|Action|Drama|Western|
+-----------------------+--------------------+


>>> CHI TIẾT CỤM SỐ 11
----------------------------------------
Từ khóa mô tả: space, visual, slow, effects, soundtrack, c, spacecraft, dull, ship, settingspacespace
10 phim mẫu tiêu biểu:


+----------------------------+----------------------+
|title                       |genres                |
+----------------------------+----------------------+
|2001: A Space Odyssey (1968)|Adventure|Drama|Sci-Fi|
+----------------------------+----------------------+


.....................................................................................


Exercise3:

In [10]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Chia tập dữ liệu 80/20
train_rec, test_rec = df_ratings.randomSplit([0.8, 0.2], seed=42)

# 2. Cấu hình và huấn luyện mô hình ALS
als = ALS(
    maxIter=10, 
    regParam=0.1, 
    userCol="userId", 
    itemCol="movieId", 
    ratingCol="rating",
    coldStartStrategy="drop", # Bỏ qua user/item mới trong tập test để tính toán không bị lỗi
    nonnegative=True
)

als_model = als.fit(train_rec)

# 3. Dự đoán trên tập Test và tính RMSE
predictions_rec = als_model.transform(test_rec)

evaluator_rmse = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse = evaluator_rmse.evaluate(predictions_rec)

print(f"MÔ HÌNH HỆ GỢI Ý (ALS) ")
print(f"Root Mean Squared Error (RMSE) trên tập Test: {rmse:.4f}")

MÔ HÌNH HỆ GỢI Ý (ALS) 
Root Mean Squared Error (RMSE) trên tập Test: 0.8764


In [16]:
from pyspark.sql.functions import col, collect_set, expr, size, array_intersect, avg

# 1. Xác định "Relevant Items" (Phim thực sự thích, rating >= 4.0) trong tập Test
relevant_test = test_rec.filter(col("rating") >= 4.0) \
    .groupBy("userId") \
    .agg(collect_set("movieId").alias("ground_truth"))

# Lọc những user có ít nhất 1 relevant item theo yêu cầu
valid_users = relevant_test.filter(size(col("ground_truth")) >= 1)

# 2. Sinh ra Top 10 gợi ý
user_subset = valid_users.select("userId")
top10_recs = als_model.recommendForUserSubset(user_subset, 10)

# Trích xuất mảng movieId
top10_recs = top10_recs.withColumn("rec_movies", expr("transform(recommendations, x -> x.movieId)"))

# 3. Tính Precision@10
eval_df = valid_users.join(top10_recs, "userId")
eval_df = eval_df.withColumn("hits", size(array_intersect(col("ground_truth"), col("rec_movies"))))
eval_df = eval_df.withColumn("precision_at_10", col("hits") / 10.0)

# Tính trung bình
avg_precision = eval_df.select(avg("precision_at_10")).collect()[0][0]


print(f"ĐÁNH GIÁ ĐỘ CHÍNH XÁC (PRECISION@10) ")
print(f"Precision@10 (Averaged across valid users): {avg_precision:.4f}")

ĐÁNH GIÁ ĐỘ CHÍNH XÁC (PRECISION@10) 
Precision@10 (Averaged across valid users): 0.0010


In [17]:
from pyspark.sql.functions import explode, desc, when, col

# 1. Chọn 3 User ID ngẫu nhiên
random_3_users = df_ratings.select("userId").distinct().sample(False, 0.1, seed=42).limit(3)

print("="*100)
print(" TOP 10 PHIM GỢI Ý CHO 3 NGƯỜI DÙNG NGẪU NHIÊN ")
print("="*100)

# 2. Lấy 10 phim gợi ý
recs_for_3 = als_model.recommendForUserSubset(random_3_users, 10)

# 3. Trải phẳng mảng gợi ý và CẮT TỈA ĐIỂM SỐ
recs_exploded = recs_for_3.select("userId", explode("recommendations").alias("rec"))
recs_final = recs_exploded.select(
    "userId", 
    col("rec.movieId").alias("movieId"), 
    # Nếu mô hình dự đoán > 5.0 thì ép về 5.0, ngược lại giữ nguyên
    when(col("rec.rating") > 5.0, 5.0).otherwise(col("rec.rating")).alias("predicted_rating")
)

# 4. Join với bảng phim để lấy thông tin chi tiết
final_output = recs_final.join(df_movies, "movieId") \
    .select("userId", "title", "genres", "predicted_rating") \
    .orderBy("userId", desc("predicted_rating"))

# In kết quả
final_output.show(30, truncate=False)

 TOP 10 PHIM GỢI Ý CHO 3 NGƯỜI DÙNG NGẪU NHIÊN 


+------+-----------------------------------------------------------------------------------------------------------------------------+---------------------------+------------------+
|userId|title                                                                                                                        |genres                     |predicted_rating  |
+------+-----------------------------------------------------------------------------------------------------------------------------+---------------------------+------------------+
|31    |Trekkies (1997)                                                                                                              |Documentary                |5.0               |
|31    |Ivan's Childhood (a.k.a. My Name is Ivan) (Ivanovo detstvo) (1962)                                                           |Drama|War                  |5.0               |
|31    |Adam's Rib (1949)                                                                 